# Gold Claim Fact

## Purpose

I use this notebook as one of the transformation sources for the Gold Lakeflow
pipeline.

I create the central claim-level fact table used by the analytical model.

### Source

`health_insurance.silver.claims`

### Pipeline target

`health_insurance.gold.fact_claim`

### Fact grain

One row in `fact_claim` represents one insurance claim.

I preserve the claim-level identifiers, dates, classifications, monetary
measures, operational measures, and fraud indicators required for downstream
analytics.

### Dimensional relationships

I generate the same deterministic keys used by:

- `dim_patient`
- `dim_provider`

This allows `fact_claim` to reference both dimensions consistently without
requiring arbitrary generated IDs.

### Modeling approach

I keep descriptive patient and provider attributes in their dimensions rather
than duplicating them throughout the fact table.

I retain claim-specific classifications such as diagnosis, procedure, service,
admission, and discharge attributes because they describe the claim event
itself.

In [0]:
# importing the Lakeflow API and Spark functions used by the Claim fact.

from pyspark import pipelines as dp
from pyspark.sql import functions as F

CATALOG = "health_insurance"

CLAIMS_SOURCE = f"{CATALOG}.silver.claims"

## Claim fact transformation

building the fact table directly from the validated Silver Claims dataset.

I generate three deterministic keys:

- `claim_key`
- `patient_key`
- `provider_key`

The Patient and Provider key formulas are identical to the formulas used in
the dimension notebook.

I also derive a small number of claim-level analytical measures that are useful
across multiple downstream Gold products.

In [0]:
# defining the pipeline-managed claim-level fact table.

@dp.materialized_view(
    name="fact_claim",
    comment="Claim-level Gold fact table with dimensional keys and analytical measures."
)
def fact_claim():

    claims_df = spark.read.table(
        CLAIMS_SOURCE
    )

    return (
        claims_df

        # generating a deterministic surrogate key for each claim.

        .withColumn(
            "claim_key",
            F.sha2(
                F.concat_ws(
                    "||",
                    F.lit("CLAIM"),
                    F.col("claim_id").cast("string")
                ),
                256
            )
        )

        # reproducing the exact Patient key formula
        # used in dim_patient.

        .withColumn(
            "patient_key",
            F.sha2(
                F.concat_ws(
                    "||",
                    F.lit("CLAIMS_PATIENT"),
                    F.col("patient_id").cast("string")
                ),
                256
            )
        )

        # reproducing the exact Provider key formula
        # used in dim_provider.

        .withColumn(
            "provider_key",
            F.sha2(
                F.concat_ws(
                    "||",
                    F.lit("CLAIMS_PROVIDER"),

                    F.coalesce(
                        F.col("hospital_id").cast("string"),
                        F.lit("UNKNOWN")
                    ),

                    F.coalesce(
                        F.col("provider_type"),
                        F.lit("UNKNOWN")
                    ),

                    F.coalesce(
                        F.col("provider_specialty"),
                        F.lit("UNKNOWN")
                    ),

                    F.coalesce(
                        F.col("provider_city"),
                        F.lit("UNKNOWN")
                    ),

                    F.coalesce(
                        F.col("provider_state"),
                        F.lit("UNKNOWN")
                    )
                ),
                256
            )
        )

        # calculating the direct patient financial responsibility.

        .withColumn(
            "patient_out_of_pocket_amount",
            (
                F.coalesce(
                    F.col("deductible_amount"),
                    F.lit(0)
                )
                +
                F.coalesce(
                    F.col("copay_amount"),
                    F.lit(0)
                )
            ).cast("decimal(18,2)")
        )

        # deriving the remaining claim amount after the
        # recorded deductible and copay amounts.

        .withColumn(
            "estimated_insurer_amount",
            F.greatest(
                F.col("claim_amount")
                -
                F.coalesce(
                    F.col("deductible_amount"),
                    F.lit(0)
                )
                -
                F.coalesce(
                    F.col("copay_amount"),
                    F.lit(0)
                ),
                F.lit(0)
            ).cast("decimal(18,2)")
        )

        # creating reporting-friendly date attributes
        # while preserving the original claim and service dates.

        .withColumn(
            "claim_year",
            F.year("claim_date")
        )

        .withColumn(
            "claim_month",
            F.month("claim_date")
        )

        .withColumn(
            "claim_year_month",
            F.date_format(
                F.col("claim_date"),
                "yyyy-MM"
            )
        )

        .select(
            # Keys
            "claim_key",
            "claim_id",
            "patient_key",
            "provider_key",

            # Business identifiers
            "patient_id",
            "policy_number",
            "hospital_id",

            # Dates
            "claim_date",
            "service_date",
            "policy_expiration_date",
            "claim_year",
            "claim_month",
            "claim_year_month",

            # Clinical / service classifications
            "diagnosis_code",
            "procedure_code",
            "service_type",
            "admission_type",
            "discharge_type",

            # Financial measures
            "claim_amount",
            "deductible_amount",
            "copay_amount",
            "patient_out_of_pocket_amount",
            "estimated_insurer_amount",

            # Utilization measures
            "number_of_procedures",
            "length_of_stay_days",
            "provider_patient_distance_miles",

            # Historical / operational measures
            "previous_claims_patient",
            "previous_claims_provider",
            "claim_submission_delay_days",

            # Analytical categories
            "claim_amount_band",

            # Flags and preserving Claim quality and fraud-label provenance.
            "claim_submitted_late",
            "is_fraudulent",
            "fraud_label_conflict",
             F.col("_source_record_count").alias("source_record_count"),

            # Silver lineage
            "_source_system",
            "_source_file",
            "_ingested_at"
        )

        .withColumn(
            "_gold_transformed_at",
            F.current_timestamp()
        )
    )

## Claim fact output

When this notebook is added to the Gold Lakeflow pipeline, it defines:

`health_insurance.gold.fact_claim`

### Grain

One row per insurance claim.

### Keys

- `claim_key`
- `patient_key`
- `provider_key`

The Patient and Provider keys use the exact same deterministic formulas as
`dim_patient` and `dim_provider`.

### Core financial measures

- claim amount
- deductible amount
- copay amount
- patient out-of-pocket amount
- estimated insurer amount

### Utilization measures

- number of procedures
- length of stay
- provider-patient distance

### Operational measures

- claim submission delay
- previous Patient claims
- previous Provider claims

### Analytical flags

- late claim submission
- historical fraud label
- fraud-label conflict indicator
- reconciled source-record count

I preserve `is_fraudulent` as the historical fraud outcome supplied by the
source dataset. I do not treat it as a fraud prediction produced by this
project.

When duplicate source records disagree on the fraud label, Silver sets
`is_fraudulent` to null and marks `fraud_label_conflict = true` rather than
arbitrarily choosing one source value.

`source_record_count` records how many Bronze source rows were reconciled into
the Gold claim.